# 1-A：Workflow，路線由程式決定

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/notebooks/1-A_workflow.ipynb)

> 📍 **對應簡報**：第 1 堂「第二站：Workflow 1-A」，投影片 **16–26**（課堂時間 0:28–0:52）

## 這本 notebook 在做什麼

我們要做一個「作業小助教」：輸入一段 Python 作業，輸出一份固定格式的回饋。
這一版的**路線是你用程式畫好的**：一定先檢查語法，語法錯就解釋錯誤，語法對才給回饋，
給完回饋還要檢查格式，不合格就退回重寫。

LangGraph 只有三個零件，全部都是你已經會的 Python：

| 你會的 Python | LangGraph | 手搖飲店 |
| --- | --- | --- |
| dict | **State** | 訂單紙 |
| 函式 | **Node** | 工作站 |
| 依序呼叫 | **Edge** | 箭頭 |
| if / else | 條件邊 | 熱飲不加冰 |
| while 迴圈 | 指回前面的邊 | 甜度不對，退回重做 |

| 段落 | 你會做什麼 | 對應投影片 |
| --- | --- | --- |
| 第 1 段 | 寫出 `graph.py`、畫出流程圖，找出每個 Node 和 Edge | 17–22 |
| 第 2 段 | 先不接 AI：換三段作業，看訂單紙一格一格被填滿 | 23–24 |
| 第 3 段 | 把 AI 放進一個節點；調「最多重試次數」和「回饋字數上限」 | 25–26 |
| 加碼 | 反例：沒說「往下接著寫」，訊息就被蓋掉 | 20 |

## 怎麼用

1. 先執行 **「0. 環境設定」**。
2. 每一段：**🤔 先寫預測 → ▶ 執行 → 👀 對照**。這本要改的只有表單欄位：**作業、最多重試次數、回饋字數上限**。
3. 跑錯或斷線：從那一段的第一格重跑就好。

In [ ]:
#@title 0. 環境設定（第一次打開、或斷線後，先執行這一格）
#@markdown **模型來源**：`colab` ＝ Colab 內建模型（免費、不用 API key）；`gemini` ＝ 講師提供的 Gemini API key（備案）
模型來源 = "colab"  #@param ["colab", "gemini"]
#@markdown 選 `gemini` 時：建議在左邊 🔑「密鑰」新增 `GOOGLE_API_KEY`；也可以把 key 貼在下面（⚠️ 貼了就不要把 notebook 分享出去）
講師_API_KEY = ""  #@param {type:"string"}
#@markdown 進階（通常不用改）：指定模型名稱，留空用預設
colab_模型名稱 = ""  #@param {type:"string"}
gemini_模型名稱 = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. 用 uv 安裝套件（版本釘死，大約 10–30 秒）
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ 用 uv 安裝 LangGraph / LangChain ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4"', shell=True, check=True)

# 2. 寫出這本 notebook 用到的共用檔案
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_檔案 = {
    'llm.py': r'''"""模型轉接頭：把「Colab 內建模型」或「講師的 Gemini API key」包成同一個 model 物件。

學生不用看這個檔案。後面所有 cell 都只寫 model.invoke(...)、model.bind_tools(...)，
不管背後是哪一家模型，用法都一樣。

- colab ：google.colab.ai.generate_text（免費帳號可用，不需要 API key）
- gemini：langchain_google_genai.ChatGoogleGenerativeAI（需要 GOOGLE_API_KEY）

generate_text 只吃一段文字、只回一段文字，不支援原生的 tool calling。
所以 ColabAIChat 做了一個小轉接：
1. 把多則訊息接成一段文字（系統指示、使用者、AI、工具結果各標一個標題）
2. 有綁工具時，在文字最後附上工具清單，請模型「需要工具就只回一行 JSON」
3. 把模型回的 JSON 解析成 tool_calls，格式和原生 tool calling 一模一樣
這剛好讓你看到：模型從頭到尾只是在「說」它想用哪個工具。
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

設定檔 = Path(__file__).with_name(".llm.json")
預設_gemini_模型 = "gemini-flash-latest"


def 估算_token(text: str) -> int:
    """粗估 token 數：中日韓文字約 1 字 1 token，其他字元約 4 個 1 token。"""
    cjk = len(re.findall(r"[⺀-鿿豈-﫿＀-￯]", text))
    return max(1, round(cjk + (len(text) - cjk) / 4))


# ---------- 把訊息接成一段文字 ----------

def _工具說明(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params})：{desc}")
    return "\n".join(lines)


def _訊息轉文字(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"【系統指示】\n{m.text}")
        elif m.type == "human":
            parts.append(f"【使用者】\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("【AI】\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"【AI】\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "工具"
            parts.append(f"【工具 {name} 的結果】\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "這一次你一定要呼叫至少一個工具。\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"這一次你一定要呼叫 {tool_choice}。\n"
        parts.append(
            "【你可以用的工具】\n"
            f"{_工具說明(tools)}\n\n"
            "【回覆規則】\n"
            f"{must}"
            "如果你需要使用工具，只回一行 JSON，不要加任何其他文字、也不要加 ``` 符號：\n"
            '{"tool_calls": [{"name": "工具名稱", "args": {"參數名稱": "參數值"}}]}\n'
            "如果不需要工具（例如已經有足夠的資訊），就直接用一般文字回答，不要輸出 JSON。"
        )
    parts.append("【AI】")
    return "\n\n".join(parts)


def _解析回覆(text: str, tools: list[dict] | None) -> AIMessage:
    """模型回的是 JSON 紙條就轉成 tool_calls，否則當成一般文字。"""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """把 google.colab.ai.generate_text 包成 LangChain 的聊天模型。"""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _訊息轉文字(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _解析回覆(text, tools)
        message.usage_metadata = {
            "input_tokens": 估算_token(prompt),
            "output_tokens": 估算_token(text),
            "total_tokens": 估算_token(prompt) + 估算_token(text),
        }
        message.response_metadata["token_count"] = "估算值（Colab 內建模型不回報用量）"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- 對外：設定與取得 model ----------

def 讀設定() -> dict:
    try:
        return json.loads(設定檔.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def 存設定(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """記住選了哪個模型來源（不存 API key）。"""
    設定檔.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or 預設_gemini_模型},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _找_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = 讀設定()
    # 上雲端時沒有設定檔：用環境變數 WORKSHOP_PROVIDER=gemini（金鑰從 GOOGLE_API_KEY 讀）
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _找_api_key()
        if not key:
            raise RuntimeError(
                "選了 gemini 但找不到 API key：請在左邊 🔑「密鑰」新增 GOOGLE_API_KEY，"
                "或在「0. 環境設定」的表單貼上講師提供的 key，再重跑那一格。"
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or 預設_gemini_模型, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai 只能在 Google Colab 上使用；不在 Colab 的話請改選 gemini。") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""工作坊 notebook 共用的小工具（學生不用看）。

- 準備()          ：每一段的第一格都會呼叫，重新連上模型、重新載入改過的檔案
- 範例作業         ：表單下拉選單裡的幾段作業
- 記下() / 印出歷程()：把每次執行的 State 存起來，最後一格一次印出來給你截圖
- 顯示流程圖()      ：把 LangGraph 畫成圖
- 存到雲端硬碟() / 下載壓縮檔()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, 估算_token  # noqa: F401  （讓 notebook 可以直接 from workshop import 估算_token）

__all__ = [
    "BASE", "準備", "範例作業", "取得作業", "記下", "印出狀態", "印出訂單紙", "印出歷程", "清除歷程",
    "模型呼叫計數器", "顯示流程圖", "存到雲端硬碟", "下載壓縮檔", "估算_token",
]

BASE = Path(__file__).resolve().parent
歷程檔 = BASE / "history.jsonl"

範例作業 = {
    "少了右括號": "print('hi'\n",
    "正確的程式": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "空白": "",
    "變數打錯字": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "縮排亂掉": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "命名不一致": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def 取得作業(名稱: str) -> str:
    """表單選「我的作業」時讀 /content/review-bot/我的作業.py，其他從範例裡拿。"""
    if 名稱.startswith("我的作業"):
        path = BASE / "我的作業.py"
        if not path.exists():
            raise FileNotFoundError("還沒有「我的作業」：請先執行上面寫著 %%writefile ... 我的作業.py 的那一格。")
        return path.read_text(encoding="utf-8")
    return 範例作業[名稱]


def 準備():
    """每一段的起點：把 review-bot 放進 import 路徑、重新載入改過的檔案、回傳 model。"""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _讓_mcp_stdio_能在notebook用()
    return get_model()


def _讓_mcp_stdio_能在notebook用() -> None:
    """mcp 的 stdio_client 會把 MCP server 的錯誤輸出接到 sys.stderr。
    Colab（Linux）的 sys.stderr 有真的檔案代號，沒問題；Windows 的 Jupyter 沒有，就改接到 log 檔。"""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State 歷程 ----------

def _可存的(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _可存的(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_可存的(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def 記下(標題: str, 內容) -> None:
    """把這一次的結果（通常是 State）存進 history.jsonl。"""
    row = {"時間": _dt.datetime.now().strftime("%H:%M:%S"), "標題": 標題, "內容": _可存的(內容)}
    with 歷程檔.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _短(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"…（還有 {len(text) - n} 字）"


def 印出狀態(state: dict, 縮排: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{縮排}{k}: （{len(v)} 則訊息）")
            for m in v:
                m = _可存的(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{縮排}  [{m['type']}] 呼叫 {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{縮排}  [{m.get('type')}] {_短(m.get('content', ''), 200)}")
        else:
            print(f"{縮排}{k}: {_短(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def 印出訂單紙(第幾步: int, state: dict, 上一步: dict | None = None, 不印: tuple = ("code",)) -> None:
    """stream_mode="values" 每一步都會給整張訂單紙；順便標出這一步新寫了哪幾格。"""
    if not 上一步:
        說明 = "起點：空白訂單紙"
    else:
        新寫入 = [k for k in state if state.get(k) != 上一步.get(k)]
        說明 = "新寫入：" + ("、".join(新寫入) if 新寫入 else "（沒有改變）")
    print(f"第 {第幾步} 步（{說明}）")
    print("  ", {k: (_短(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in 不印})


def 印出歷程() -> None:
    if not 歷程檔.exists():
        print("還沒有任何紀錄：先跑上面幾段，再回來執行這一格。")
        return
    rows = [json.loads(line) for line in 歷程檔.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 共 {len(rows)} 筆紀錄（存在 {歷程檔}）\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['時間']}] {row['標題']} =====")
        內容 = row["內容"]
        if isinstance(內容, dict):
            印出狀態(內容)
        elif isinstance(內容, list):
            for j, item in enumerate(內容, 1):
                print(f"  --- 第 {j} 步 ---")
                if isinstance(item, dict):
                    印出狀態(item, "    ")
                else:
                    print(f"    {_短(item)}")
        else:
            print(f"  {_短(內容, 1000)}")
        print()


def 清除歷程() -> None:
    歷程檔.unlink(missing_ok=True)


# ---------- 數模型被呼叫了幾次 ----------

class 模型呼叫計數器(BaseCallbackHandler):
    """放進 config={"callbacks": [計數器]}，每呼叫一次模型就 +1。"""

    def __init__(self):
        self.次數 = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.次數 += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.次數 += 1


# ---------- 畫流程圖 ----------

def 顯示流程圖(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("（畫圖服務暫時連不上，改用文字版流程圖；可以貼到 https://mermaid.live 看圖）\n")
        print(g.draw_mermaid())


# ---------- 保存 ----------

def 存到雲端硬碟() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    目的地 = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, 目的地, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ 已存到雲端硬碟：{目的地}")
    print("   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）")


def 下載壓縮檔() -> None:
    from google.colab import files

    名稱 = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(名稱), "zip", BASE.parent, BASE.name)
    print(f"✅ 已打包：{zip_path}")
    files.download(zip_path)
''',
}
for rel, content in _檔案.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# 剛寫好的檔案要讓 Python 重新找一次（如果之前先跑過別的格子，Python 會記得「這個資料夾不存在」）
import importlib
importlib.invalidate_caches()

# 3. 記住模型來源（不會存 API key），測試連線
import llm
llm.存設定(模型來源, colab_模型名稱.strip(), gemini_模型名稱.strip())
if 講師_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = 講師_API_KEY.strip()
from workshop import *
model = 準備()

回覆 = model.invoke("用一句話跟剛學 Python 的同學打招呼").text
print(f"✅ 環境設定完成，模型來源：{模型來源}")
print(f"🤖 模型說：{回覆.strip()}")

### ✏️（選用）放進你自己的作業

課後作業要把**你自己寫過的一段作業**丟進來跑。把下面那格的程式換成你的作業、執行它，
之後在表單的「作業」選 **我的作業** 就會用它。課堂上可以先跳過這格。

In [ ]:
%%writefile /content/review-bot/我的作業.py
# 把這裡換成你自己寫過的一段 Python 作業，改完記得執行這一格
def average(scores):
    total = 0
    for s in scores:
        total = total + s
    return total / len(scores)

print(average([90, 85, 77]))


---
## 第 1 段：LangGraph 就是把函式用箭頭接起來

> 📍 **對應簡報**：投影片 17「把函式用箭頭接起來」、18「State」、19「Node」、21「Edge」、22「一張表記住三個零件」

下一格會把作業小助教的程式存成 `/content/review-bot/graph.py`（不用改，直接執行）。
想看程式的話，重點只有三種東西：

- **State**：`class State(TypedDict)` 那一段，就是訂單紙有哪幾格（`code`、`syntax_ok`、`feedback`、`retries`…）
- **Node**：`def check_syntax(state)` 這種函式，收訂單紙、**只回傳自己改的那幾格**
- **Edge**：最下面 `add_edge`（做完 A 一定接 B）和 `add_conditional_edges`（看情況走哪條路，就是 if）

### 🤔 預測題
1. 流程圖上會有幾個 Node（工作站）？
2. 哪幾條箭頭是「條件邊」（會分岔的）？
3. 有沒有一條箭頭是往回指的？它像 Python 的哪個語法？

In [ ]:
%%writefile /content/review-bot/graph.py
"""1-A：作業小助教的 workflow，路線由程式決定。

START → check_syntax ─語法 OK→ give_feedback → check_format ─格式 OK→ END
                     └語法錯→ explain_error → END     └不合格、retries 還沒到上限→ 退回 give_feedback

三個零件：
- State：一張訂單紙（dict），從頭傳到尾
- Node ：一個工作站（函式），收 State、只回傳自己改的那幾格
- Edge ：工作站之間的箭頭；條件邊就是一個回傳「下一站名字」的 if
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # 學生交的作業
    syntax_ok: bool | None # 語法對不對（還沒檢查時是 None）
    error: str             # 語法錯在哪裡
    feedback: str          # 給學生的回饋
    format_ok: bool | None # 回饋格式合不合格
    format_problem: str    # 不合格的原因（退回重寫時會告訴 AI）
    retries: int           # 回饋寫了幾次


# ---------- Node 1：check_syntax（純 Python） ----------

_錯誤翻譯 = [
    (r"'(.)' was never closed", lambda m: f"少了右括號 {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "字串少了結尾的引號"),
    (r"expected ':'", lambda m: "少了冒號 :"),
    (r"unexpected indent", lambda m: "縮排多了"),
    (r"expected an indented block", lambda m: "這裡需要縮排"),
    (r"unindent does not match", lambda m: "縮排對不齊"),
    (r"unmatched '(.)'", lambda m: f"多了一個 {m.group(1)}"),
    (r"invalid character", lambda m: "有全形或看不懂的字元"),
    (r"Perhaps you forgot a comma", lambda m: "可能少了逗號"),
    (r"invalid syntax", lambda m: "語法不正確"),
]


def 翻譯語法錯誤(msg: str) -> str:
    for pattern, 說法 in _錯誤翻譯:
        m = re.search(pattern, msg)
        if m:
            return 說法(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "作業是空白的，沒有任何程式"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"第 {e.lineno} 行{翻譯語法錯誤(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- 條件邊 1：語法對 → 給回饋；語法錯 → 解釋錯誤 ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2：explain_error（純 Python） ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"語法：{state['error']}\n改進：先把這一行修好，再執行一次看看。"}


# ---------- Node 3：give_feedback（兩個版本） ----------

def give_feedback_固定文字(state: State) -> dict:
    """先不接 AI 的版本：回固定的一段話。"""
    return {
        "feedback": "優點：語法正確，程式可以執行。\n改進：（這一版還沒接 AI，先用固定文字）",
        "retries": state.get("retries", 0) + 1,
    }


def 建立_give_feedback(model, 回饋字數上限: int):
    """接上 AI 的版本：AI 只是住在這個節點裡的一行程式。"""

    def give_feedback(state: State) -> dict:
        prompt = (
            "你是 Python 作業小助教，請用繁體中文給下面這段作業回饋。\n"
            "格式規定（一定要遵守）：\n"
            "- 只寫兩段：第一段以「優點：」開頭，第二段以「改進：」開頭\n"
            f"- 兩段加起來不超過 {回饋字數上限} 個字\n\n"
            f"作業：\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\n你上一版的回饋被退回，原因：{state['format_problem']}。請修正後重寫。"
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4：check_format（純 Python 的檢查站） ----------

def 建立_check_format(回饋字數上限: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        問題 = []
        if "優點" not in feedback:
            問題.append("少了「優點」段落")
        if "改進" not in feedback:
            問題.append("少了「改進」段落")
        if len(feedback) > 回饋字數上限:
            問題.append(f"太長了：{len(feedback)} 字，上限 {回饋字數上限} 字")
        return {"format_ok": not 問題, "format_problem": "；".join(問題)}

    return check_format


# ---------- 條件邊 2：格式 OK 或重寫次數到了 → 結束；否則退回重寫 ----------

def 建立_route_after_format(最多重試次數: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= 最多重試次數:
            return END  # 重寫次數到了就停，不然會無限轉圈
        return "give_feedback"

    return route_after_format


# ---------- 把零件接起來 ----------

def 建立小助教(model=None, 最多重試次數: int = 3, 回饋字數上限: int = 150):
    """model=None 時 give_feedback 用固定文字（先不接 AI）。"""
    give_feedback = 建立_give_feedback(model, 回饋字數上限) if model else give_feedback_固定文字

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", 建立_check_format(回饋字數上限))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", 建立_route_after_format(最多重試次數), ["give_feedback", END])
    return builder.compile()


def 空白訂單紙(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}


In [ ]:
#@title ▶ 執行：畫出作業小助教的流程圖
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import graph

小助教 = graph.建立小助教()        # 先不給 model：give_feedback 用固定文字
顯示流程圖(小助教)

g = 小助教.get_graph()
print("Node（工作站）：", [n for n in g.nodes if n not in ("__start__", "__end__")])
print("Edge（箭頭）：")
for e in g.edges:
    print(f"  {e.source} → {e.target}{'   ← 條件邊（if）' if e.conditional else ''}")

### 👀 你看到了什麼

- 4 個工作站：`check_syntax`、`explain_error`、`give_feedback`、`check_format`（`__start__`、`__end__` 是起點和終點）。
- 條件邊有兩處：`check_syntax` 之後分成「語法 OK → give_feedback」和「語法錯 → explain_error」；`check_format` 之後分成「OK → 結束」和「不合格 → 退回 give_feedback」。
- `check_format → give_feedback` 那條**往回指**的箭頭，就像 `while` 迴圈：甜度不對，退回重做。
- 🏅 加分題：展開上面 `graph.py` 那一格，找出這兩條條件邊分別寫在哪一行（提示：搜尋 `add_conditional_edges`）。

---
## 第 2 段：先不接 AI，三段作業先猜再跑

> 📍 **對應簡報**：投影片 23「先不接 AI：三段作業，先猜再跑」、投影片 24「資料流：看訂單紙一格一格被填滿」

這一段的節點都是普通 Python，沒有 AI。下面的表單有一個「作業」下拉選單，
**請依序選「少了右括號」「正確的程式」「空白」各執行一次**。
程式用 `stream_mode="values"` 印出每一步的**整張訂單紙**，並標出這一步新寫了哪幾格。

### 🤔 學習單（先寫在紙上，再執行比對）
1. 輸入 `print("hi"`（少了右括號），會經過哪些節點？`syntax_ok` 最後是什麼？
2. 輸入正確的程式，`feedback` 在哪一步之後才有內容？
3. 輸入空白，會發生什麼事？

In [ ]:
#@title ▶ 執行：看訂單紙一格一格被填滿（先不接 AI）
作業 = "少了右括號"  #@param ["少了右括號", "正確的程式", "空白", "變數打錯字", "我的作業"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import graph

小助教 = graph.建立小助教()        # 不給 model：give_feedback 用固定文字
code = 取得作業(作業)
print(f"作業：{code!r}\n")

步驟們, 上一步 = [], None
for 第幾步, state in enumerate(小助教.stream(graph.空白訂單紙(code), stream_mode="values"), start=1):
    印出訂單紙(第幾步, state, 上一步)
    步驟們.append(state)
    上一步 = state

print("\n最後的回饋：")
print(state["feedback"])
記下(f"1-A 第 2 段：{作業}（不接 AI）", 步驟們)

### 👀 你看到了什麼

1. **少了右括號**：`check_syntax` 寫入 `syntax_ok: False` 和錯在哪一行 → 條件邊走 `explain_error` → 寫入 `feedback` → 結束。完全沒經過 `give_feedback`。
2. **正確的程式**：`syntax_ok: True` → `give_feedback` 寫入 `feedback` 和 `retries: 1`（第 3 步之後才有回饋）→ `check_format` 寫入 `format_ok: True` → 結束。
3. **空白**：空白其實是「合法」的 Python，但我們在 `check_syntax` 裡多寫了一個 `if`：空白就當成沒交作業。這就是「能寫成 if 的事，寫進程式」。
- 每一步只有一兩格改變：**每個節點只寫自己負責的那幾格**，其他格原封不動往下傳。
- 以後看任何 LangGraph 程式，先問兩件事：State 有哪幾格？每個節點寫哪一格？

---
## 第 3 段：把 AI 放進一個節點

> 📍 **對應簡報**：投影片 25「把 AI 放進一個節點」、投影片 26「三種設計模式，這張流程圖全都有」

現在把 `give_feedback` 換成呼叫模型的版本（在 `graph.py` 的 `建立_give_feedback`）。
AI 只是住在某個節點裡的一行程式：**流程圖沒變、State 沒變、箭頭沒變**。

`check_format` 是一個純 Python 的**檢查站**：回饋一定要有「優點」和「改進」兩段、不能超過字數上限，
不合格就退回 `give_feedback` 重寫，重寫到「最多重試次數」就停。

### 🤔 預測題（先寫下來，再改表單、執行）
1. 「回饋字數上限」從 150 改成 **30**，`retries` 會變多還是變少？`format_ok` 最後會是什麼？
2. 「最多重試次數」改成 **1**，字數上限 30 時會發生什麼事？
3. 同一段作業跑兩次，回饋的文字會一樣嗎？格式呢？

In [ ]:
#@title ▶ 執行：AI 寫回饋 ＋ 程式檢查格式（不合格退回重寫）
作業 = "變數打錯字"  #@param ["少了右括號", "正確的程式", "空白", "變數打錯字", "我的作業"]
最多重試次數 = 3  #@param {type:"slider", min:1, max:5, step:1}
回饋字數上限 = 150  #@param {type:"slider", min:20, max:300, step:10}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import graph

小助教 = graph.建立小助教(model, 最多重試次數=最多重試次數, 回饋字數上限=回饋字數上限)
code = 取得作業(作業)

步驟們, 上一步 = [], None
for 第幾步, state in enumerate(小助教.stream(graph.空白訂單紙(code), stream_mode="values"), start=1):
    印出訂單紙(第幾步, state, 上一步)
    步驟們.append(state)
    上一步 = state

print(f"\n最後：retries = {state['retries']}，format_ok = {state['format_ok']}")
if state.get("format_problem"):
    print(f"最後一次不合格的原因：{state['format_problem']}")
print("\n最後的回饋：")
print(state["feedback"])
記下(f"1-A 第 3 段：{作業}（AI，重試上限 {最多重試次數}、字數上限 {回饋字數上限}）", 步驟們)

### 👀 你看到了什麼

- 字數上限越小，AI 越容易寫太長被退回，`retries` 就越多；到了「最多重試次數」就停，這時 `format_ok` 可能還是 `False`。
  **不設上限的話，它可能永遠轉下去**：重寫最多幾次，是一定要鎖死在程式裡的事。
- 同一段作業跑兩次，回饋的**文字**會不同（AI 每次都重新想），但**格式**一定有兩段、一定不超過上限，因為檢查是程式做的。
- 這張流程圖剛好有 Anthropic〈Building effective agents〉的三種設計模式：
  - **串接（prompt chaining）**：`check_syntax → give_feedback → check_format` 固定順序的小步驟
  - **分流（routing）**：`check_syntax` 之後的分叉，先判斷類型再交給專門處理的步驟
  - **評估與修正（evaluator-optimizer）**：`check_format` 退回 `give_feedback` 的圈，產出、檢查、不合格就重做

---
## 加碼：沒說「這一格要往下接著寫」，訊息就被蓋掉

> 📍 **對應簡報**：投影片 20「反例：訊息被蓋掉」（必踩的雷 2）

還記得 1-0 第 1 段嗎？模型沒有記憶，對話要整段貼回去。在 LangGraph 裡，對話放在 State 的 `messages` 那一格。
如果沒有告訴 LangGraph「這一格要往下接著寫（`add_messages`）」，它預設會**整格覆蓋**。

### 🤔 預測題
輸入一則訊息「我叫小明」，經過一個會呼叫模型的節點之後，`messages` 裡會有幾則訊息？兩種寫法各是幾則？

In [ ]:
#@title ▶ 執行：BadState（覆蓋） vs 有 add_messages（接著寫）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

class BadState(TypedDict):
    messages: list                               # ← 預設行為是「覆蓋」

class GoodState(TypedDict):
    messages: Annotated[list, add_messages]      # ← 告訴 LangGraph：這一格要往下接著寫

def chatbot(state) -> dict:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

for 名稱, StateClass in [("BadState（沒寫 add_messages）", BadState), ("GoodState（有 add_messages）", GoodState)]:
    builder = StateGraph(StateClass)
    builder.add_node("chatbot", chatbot)
    builder.add_edge(START, "chatbot")
    builder.add_edge("chatbot", END)
    結果 = builder.compile().invoke({"messages": [{"role": "user", "content": "我叫小明"}]})
    print(f"{名稱}：messages 有 {len(結果['messages'])} 則")
    for m in 結果["messages"]:
        print(f"   [{m.type if hasattr(m, 'type') else m['role']}] {(m.text if hasattr(m, 'text') else m['content'])[:60]}")
    記下(f"1-A 加碼：{名稱}", 結果)

### 👀 你看到了什麼

- `BadState` 只剩 **1 則**：使用者說的「我叫小明」被模型的回覆蓋掉了。
- 加上 `Annotated[list, add_messages]` 就有 **2 則**：使用者的話和模型的回覆都在。
- 怎麼讀這行：`messages` 是一個 list，但 LangGraph 處理時不要覆蓋，要用 `add_messages` 合併。口訣：**`Annotated` 第一格是真型別，第二格是給框架看的提示。**
- 1-B 會用到的 `MessagesState`，裡面就是這一行。

## 收尾：存檔與交作業

- 💾 **存到雲端硬碟**：把 `/content/review-bot/`（你改過的手冊、程式、State 紀錄）存一份到自己的 Google 雲端硬碟。
- ⬇️ **下載 zip**：直接下載到電腦。
- 📸 **最後一格**：印出這本 notebook 跑過的所有 State，截圖就能交作業。

Colab 斷線後 `/content/` 裡的檔案會消失，想留下來就先存。notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」。

In [ ]:
#@title 💾 存到 Google 雲端硬碟（會跳出授權視窗）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
存到雲端硬碟()

In [ ]:
#@title ⬇️ 下載 zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
下載壓縮檔()

In [ ]:
#@title 📸 印出整份 State 歷程（截圖交作業）
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
印出歷程()